# Exact 2x2 lattice: a bounded reverse-kernel reference

This notebook has been executed using NumPy. It is self-contained and needs no
checkpoint, Torch, author installation, or project path. Running it again only
recomputes deterministic arrays in memory. It does not write to the user's project.

## Scientific scope

This is an **independent mathematical reference**, not a reproduction of the paper
or the user's 20x20 lattice distribution. The clean law has 16 spin configurations,
with probability proportional to exp(0.5 * sum of four open-square neighbor products).
No spin-site independence is assumed. The interaction coefficient is illustrative,
not calibrated from the user's data. Noise variance T=5 and mean p=-1,0,+1 are
external fixed conditions; p changes the noise law, not this chosen clean law.

Two adjacent pairs of the recorded author schedule are examined: 99->98 and 98->97.
At each pair and noise mean, evaluate three deterministic observation vectors:
the marginal mean, a one-marginal-SD uniform offset, and a one-marginal-SD
checkerboard offset. These are conditional point diagnostics, not integrations
over all observations or a full sampling benchmark. Choices are illustrative,
not a preregistered validation. No confidence intervals or Monte Carlo are used.

The reference forward coupling is

X_t = q X_s + p (sqrt(1-a_t) - q sqrt(1-a_s)) + sqrt(T(1-q^2)) Z,
where q=sqrt(a_t/a_s), Z is a standard independent four-dimensional Gaussian,
and the clean/noise marginal is X_t=sqrt(a_t)U+sqrt(1-a_t)epsilon.

For each observation x, Bayes' rule enumerates all 16 posterior clean-state weights.
Conditioning additionally on each U gives a Gaussian reverse bridge. Summing those
16 weighted bridges yields the exact reverse conditional mixture for this coupling.
An ideal predictor here means E[epsilon|x], not knowledge of the realized epsilon.

## Comparisons

1. Literal eta=1 author-style update with an ideal population predictor.
2. That update with its existing deterministic mean correction.
3. A Gaussian matching the exact conditional mean and full covariance, used only
   to assess whether these moments identify the mixture (not a proposed sampler).

The first two have conditional covariance c1^2 T I. The exact mixture adds
h^2 Cov(U|x), with h=sqrt(a_s)-q(1-a_s)/(1-a_t)*sqrt(a_t).
The third matches two moments by definition. A nonzero projected fourth cumulant
or characteristic-function difference shows that it need not match the distribution.

These facts do not establish the author's intended temporal coupling, evaluate
auxiliary-channel steering, explain all checkpoint error, or authorize installing
a correction into the author's code. Conditional-variance error may be a small
finite-step approximation; its numerical size must be reported alongside existence.
At the tested high-noise observations, the non-Gaussian discrepancy is small.

## Schedule provenance

Only the three needed alpha values are embedded, copied from metrics.json in
the user's time_profile_results.zip, produced by the previously hash-checked
PyTorch 2.5.1+cpu source run. The whole 100-step schedule is not regenerated here.
Archive SHA256: `78519a5dbdac8034b8084392be24c15f6094ac61d60ed9c2f518f79deda0e599`.


In [1]:
import itertools
import numpy as np

# Exact float32 schedule values recorded in the user's pinned-source run.
ALPHAS = {97: 0.039609555155038834, 98: 0.01990976370871067, 99: 1.9909764887415804e-05}
J = 0.5  # Dimensionless interaction in this reference distribution only.
T = 5.0
PAIRS = [(99, 98), (98, 97)]
PRESSURES = [-1.0, 0.0, 1.0]  # Noise means, not an identified physical pressure.
# Open 2x2 square: exactly four undirected bonds, no periodic double counting.
EDGES = [(0, 1), (0, 2), (1, 3), (2, 3)]
states = np.asarray(list(itertools.product([-1.0, 1.0], repeat=4)))
log_prior = J * sum(states[:, i] * states[:, j] for i, j in EDGES)
log_prior -= np.max(log_prior)
prior = np.exp(log_prior)
prior /= prior.sum()
assert len(states) == 16 and np.isclose(prior.sum(), 1.0)
assert np.allclose(prior @ states, 0.0, atol=1e-14)

offsets = {
    'center': np.zeros(4),
    'uniform_1sd': np.ones(4),
    'checkerboard_1sd': np.array([1.0, -1.0, -1.0, 1.0]),
}
magnetization_direction = np.ones(4) / 2  # Unit norm, not mean magnetization.
rows = []
details = {}
max_identity_error = 0.0
for t, s in PAIRS:
    a, b = ALPHAS[t], ALPHAS[s]
    rt, rs = np.sqrt(a), np.sqrt(b)
    st, ss = np.sqrt(1-a), np.sqrt(1-b)
    q = rt/rs
    K = q * (1-b)/(1-a)
    h = rs - K*rt
    c1 = np.sqrt((1-a/b)*(1-b)/(1-a))
    c2 = np.sqrt((1-b)-c1*c1)
    bridge_variance = c1*c1*T
    # Independent conditional-Gaussian calculation given each clean state.
    forward_innovation_variance = T*(1-q*q)
    independent_variance = (1-b)*T * forward_innovation_variance / ((1-a)*T)
    assert np.isclose(bridge_variance, independent_variance, rtol=1e-12, atol=1e-12)
    assert np.isclose(c2, K*st, rtol=1e-12, atol=1e-12)
    for p in PRESSURES:
        for query, offset in offsets.items():
            # Each coordinate has prior variance 1 in this symmetric spin law.
            x = st*p + np.sqrt(a+(1-a)*T)*offset
            forward_means = rt*states + st*p
            log_w = np.log(prior) - np.sum((x-forward_means)**2, axis=1)/(2*(1-a)*T)
            log_w -= np.max(log_w)
            w = np.exp(log_w)
            w /= w.sum()
            clean_mean = w @ states
            centered_states = states - clean_mean
            clean_covariance = centered_states.T @ (w[:,None]*centered_states)

            # Exact reverse law is this 16-component mixture, not one Gaussian.
            bridge_means = rs*states + ss*p + K*(x-forward_means)
            exact_mean = w @ bridge_means
            centered_means = bridge_means - exact_mean
            exact_covariance = bridge_variance*np.eye(4) + centered_means.T @ (w[:,None]*centered_means)

            # Population noise predictor: no access to a realized latent state.
            noise_mean = (x-rt*clean_mean)/st
            original_mean = rs*clean_mean + c2*noise_mean + c1*p
            mean_corrected = original_mean + (ss-c1-c2)*p
            original_covariance = bridge_variance*np.eye(4)
            missing_covariance = h*h*clean_covariance
            err = max(np.max(np.abs(mean_corrected-exact_mean)),
                      np.max(np.abs(exact_covariance-original_covariance-missing_covariance)))
            max_identity_error = max(max_identity_error, float(err))
            assert err < 1e-12
            assert np.linalg.eigvalsh(missing_covariance).min() > -1e-12

            # Matching full covariance still need not match the mixture law.
            # Fourth cumulant along the normalized all-ones direction.
            component_offsets = centered_means @ magnetization_direction
            between_variance = float(w @ component_offsets**2)
            projected_variance = bridge_variance + between_variance
            fourth = float(w @ component_offsets**4 + 6*bridge_variance*between_variance + 3*bridge_variance**2)
            fourth_cumulant = fourth - 3*projected_variance**2
            # A same-mean, same-covariance Gaussian has fourth cumulant zero.
            frequency = 1/np.sqrt(projected_variance)
            exact_cf = np.exp(-.5*frequency**2*bridge_variance) * (w @ np.exp(1j*frequency*component_offsets))
            matched_gaussian_cf = np.exp(-.5*frequency**2*projected_variance)
            row = dict(t=t, s=s, noise_mean=p, query=query,
                max_abs_original_mean_error=float(np.max(np.abs(original_mean-exact_mean))),
                max_abs_mean_corrected_error=float(np.max(np.abs(mean_corrected-exact_mean))),
                component_variance=float(bridge_variance),
                missing_covariance_trace=float(np.trace(missing_covariance)),
                missing_neighbor_covariance=float(missing_covariance[0,1]),
                projected_fourth_cumulant=float(fourth_cumulant),
                projected_excess_kurtosis=float(fourth_cumulant/projected_variance**2),
                matched_gaussian_characteristic_difference=float(abs(exact_cf-matched_gaussian_cf)))
            rows.append(row)
            details[(t,p,query)] = dict(weights=w, bridge_means=bridge_means,
                clean_covariance=clean_covariance, exact_mean=exact_mean,
                exact_covariance=exact_covariance, missing_covariance=missing_covariance)

assert len(rows) == 18
for t, s in PAIRS:
    centered = details[(t,0.0,'center')]
    assert np.allclose(centered['weights'], prior, atol=1e-14)
    for query in offsets:
        group = [r for r in rows if r['t']==t and r['query']==query]
        assert np.ptp([r['missing_covariance_trace'] for r in group]) < 1e-12

print('PASS: 16-state enumeration, normalization, bridge identities, covariance decomposition, shift invariance.')
print(f'Maximum identity discrepancy: {max_identity_error:.3e}')
print('No neural model, training, Monte Carlo, source changes, or full reverse chain.')
print('t->s | p | query | original mean error | missing covariance trace | projected excess kurtosis')
for r in rows:
    print(f"{r['t']}->{r['s']} | {r['noise_mean']:+.0f} | {r['query']:18s} | "
          f"{r['max_abs_original_mean_error']:.8f} | {r['missing_covariance_trace']:.8f} | "
          f"{r['projected_excess_kurtosis']:+.8f}")
print('Full conditional mixture parameters are available in details; metrics are in rows.')


PASS: 16-state enumeration, normalization, bridge identities, covariance decomposition, shift invariance.
Maximum identity discrepancy: 7.550e-15
No neural model, training, Monte Carlo, source changes, or full reverse chain.
t->s | p | query | original mean error | missing covariance trace | projected excess kurtosis
99->98 | -1 | center             | 0.03050821 | 0.07948302 | -0.00015229
99->98 | -1 | uniform_1sd        | 0.03050821 | 0.07948107 | -0.00015227
99->98 | -1 | checkerboard_1sd   | 0.03050821 | 0.07948299 | -0.00015229
99->98 | +0 | center             | 0.00000000 | 0.07948302 | -0.00015229
99->98 | +0 | uniform_1sd        | 0.00000000 | 0.07948107 | -0.00015227
99->98 | +0 | checkerboard_1sd   | 0.00000000 | 0.07948299 | -0.00015229
99->98 | +1 | center             | 0.03050821 | 0.07948302 | -0.00015229
99->98 | +1 | uniform_1sd        | 0.03050821 | 0.07948107 | -0.00015227
99->98 | +1 | checkerboard_1sd   | 0.03050821 | 0.07948299 | -0.00015229
98->97 | -1 | center    

## Interpretation

All finite-state checks passed. At centered observations the posterior spin law
equals the chosen clean law, so neighbor covariance is retained exactly.

| Pair | Original mean error for p=+1, per site | Original per-site conditional variance | Missing per-site variance | Missing neighbor covariance |
|---|---:|---:|---:|---:|
| 99->98 | 0.03050821 | 4.89564820 | 0.01987076 | 0.01065755 |
| 98->97 | 0.40588878 | 2.43676381 | 0.01019979 | 0.00547059 |

The mean correction matches the exact reference conditional mean within numerical
precision but leaves the covariance deficit, including off-diagonal correlations.
Even a full-covariance matched Gaussian is not exactly the conditional mixture:
its projected fourth cumulant is zero while the enumerated mixture's is not.
In these tested high-noise cases, that higher-order discrepancy is small; this is
not evidence that it dominates the large observed neural-checkpoint failures.

No corrected sampler has been installed, and no complete reverse chain was run.
This notebook supplies an exact conditional reference, not a claim of successful
lattice generation or paper reproduction. The next decision should distinguish
reproducing the published implementation from designing an independent method.
